# Kalshi vs Polymarket: how big must a gap get before it closes, and how fast does it close?

This notebook reads the 1-minute gap series made by `pull_all_events.ipynb`, fits
the project's threshold model to each of the nine markets, and then checks
whether the fit can be trusted.

* **Part A — Results:** the model, the fit, and what it says.
* **Part B — Validation:** (1) does the answer depend on the time step,
  (2) does the method recover known values from simulated data,
  (3) does the c = 0 case match the textbook formulas, and
  (4) does the model predict better than simpler models on data it hasn't seen?
* **Part C — Appendix:** robustness checks, topics, liquidity.

### The model

`G(t)` is the gap: Kalshi price − Polymarket price (0.01 = 1 cent).

```
G(t+Δ) − G(t) = −κ · max(0, |G(t)| − c) · sign(G(t)) + noise
```

* **c — the no-trade band.** Closing a gap means buying on one platform and
  selling on the other, which costs spreads and fees. Gaps smaller than c aren't
  worth trading, so they drift. c is read as the **implied transaction cost**.
* **κ — the pull.** Beyond c, the excess shrinks by the fraction κ per step Δ.
  Results are reported as a **half-life in minutes**: how long the excess
  takes to halve.

### Choices and why

| Choice | Value | Why |
|---|---|---|
| Time step Δ | **5 minutes** | At 1 minute, quotes that flicker and immediately revert look like very fast gap closing. Figure 3 compares 1, 5 and 15 minutes. |
| What the gap is measured from | **The market's median gap** (one fixed value per market) | Markets have a steady offset between the platforms (in the 2024 election Polymarket sat ~3¢ above Kalshi); measured from zero, the offset would look like a wide band. A *trailing* median would also follow a drifting offset, but it can't be used to estimate c: inside the band the gap wanders freely, a trailing median follows that wandering, and the band disappears — Figure 7 shows a known 2–3¢ band being estimated as ~0¢ that way. The cost: when the offset really drifts (the Super Bowl's median gap went from +0.6¢ to −0.65¢), a fixed centre fits that market poorly (Check 4b). |
| Pairs | Rows exactly Δ apart, every minute as a start | No resampling or filling. Overlapping pairs are fine because the uncertainty comes from a block bootstrap. |
| Flagged minutes | Left out | One-minute spikes flagged by the pull notebook (they reverse immediately) would otherwise dominate the fit for large gaps. Appendix A1 shows the fit with them. |
| Candidate c | 0 to 10¢ in 0.1¢ steps | The best c is the one with the smallest squared error. |
| Minimum data beyond c | 300 pairs | So a handful of extreme minutes can't decide c. |
| Uncertainty | Block bootstrap, 1-hour blocks, 1,000 draws | Resampling whole hours keeps the dependence between nearby minutes; c and κ are refitted on every draw. |
| Out-of-sample split | First 70% of each market's time span to fit, last 30% to test | κ, c and the median gap all come from the first 70% only, so nothing from the test period leaks in. |

### Data

The notebook looks for the `results/` folder (or the `out/` folder from a pull
run) next to it or in `/content`. If it can't find one in Colab, it asks you to
**upload `results.zip`** (a zip of the repo's `results/` folder). The data must
be the 1-minute results from the fixed pull notebook.

**Run all cells top to bottom** (about 3–6 minutes; the simulations take most of
it). Tables and figures are also saved to `fit_out/`.

In [ ]:
%pip install -q pandas numpy matplotlib

## Settings and helpers

In [ ]:
import time
import textwrap
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

T_START = time.time()

STEP = 5                                           # main time step (minutes)
CENTRE = "median"                                  # gaps measured from the market's median gap (see "Choices and why")
STEPS_TO_COMPARE = (1, 5, 15)                      # Figure 3
C_GRID = np.round(np.arange(0, 0.1001, 0.001), 4)  # candidate c: 0 to 10 cents in 0.1-cent steps
MIN_BEYOND = 300                                   # a c needs at least this many pairs with |gap| > c
N_BOOT = 1000                                      # bootstrap draws for the main fit
BLOCK_MINUTES = 60                                 # bootstrap block length
TRAIN_SHARE = 0.7                                  # out-of-sample: first 70% of each market's span
N_SIM = 100                                        # simulated replicates per market
N_BOOT_SIM = 200                                   # bootstrap draws per simulated replicate
KNOWN_C_CENTS = (0, 1, 2, 3)                       # known-c test
N_SIM_KNOWN = 50
TICK = 0.005                                       # half a cent: the smallest step in the gap
SEED = 215

# Kalshi-side trading cost, the benchmark for c: half the bid-ask spread + Kalshi's
# taker fee, approx. KALSHI_FEE_RATE * p * (1 - p) per contract. 0.07 is Kalshi's
# standard rate; check the current fee schedule if exact fees matter.
KALSHI_FEE_RATE = 0.07

OUT = Path("fit_out")
OUT.mkdir(exist_ok=True)
rng = np.random.default_rng(SEED)

CATEGORY_ORDER = ["Politics", "Sports", "Culture"]
CATEGORY_COLORS = {"Politics": "#2a78d6", "Sports": "#eb6834", "Culture": "#1baf7a"}  # colorblind-safe
CATEGORY_MARKERS = {"Politics": "o", "Sports": "s", "Culture": "^"}
INK, MUTED, BAND = "#222222", "#666666", "#e9e9e9"

SHORT_NAMES = {
    "election_2024_DJT_wins": "2024 election",
    "fomc_sep2026_hike_25bp": "Sept 2026 FOMC",
    "shutdown_oct2025_yes": "Oct 2025 shutdown",
    "super_bowl_lx_seattle_wins": "Super Bowl LX",
    "world_series_2025_dodgers_win": "2025 World Series",
    "nba_finals_2025_okc_win": "2025 NBA Finals",
    "papal_conclave_2025_prevost": "Papal conclave",
    "oscars_2025_anora_best_pic": "2025 Oscars",
    "taylor_swift_engagement_yes": "Swift engagement",
}

plt.rcParams.update({
    "figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": "#e6e6e6", "grid.linewidth": 0.8, "axes.axisbelow": True,
    "axes.edgecolor": "#888888", "axes.titleweight": "bold", "font.size": 10,
})


def caption(fig, text, width=150, y=-0.02):
    # A plain-language "how to read this" box under a figure.
    fig.text(0.01, y, textwrap.fill(text, width), ha="left", va="top", fontsize=9.5, color="#333333",
             bbox=dict(boxstyle="round,pad=0.6", fc="#f6f6f4", ec="#dddddd"))


def category_handles(extra=()):
    h = [plt.Line2D([], [], color=CATEGORY_COLORS[c], marker=CATEGORY_MARKERS[c], linestyle="",
                    markersize=8, label=c) for c in CATEGORY_ORDER]
    return h + list(extra)


def plain_log(ax, axis="x", subs=(1.0, 2.0, 5.0)):
    # Log scale with ordinary numbers (0.01, 0.1, 1) instead of 10^-2 style labels.
    from matplotlib.ticker import FuncFormatter, LogLocator, NullFormatter
    (ax.set_xscale if axis == "x" else ax.set_yscale)("log")
    a = ax.xaxis if axis == "x" else ax.yaxis
    a.set_major_locator(LogLocator(base=10, subs=subs))
    a.set_major_formatter(FuncFormatter(lambda v, _: f"{v:g}"))
    a.set_minor_formatter(NullFormatter())


def dodge_labels(ax, texts, pad_px=2, max_iter=200):
    # Nudges annotation labels apart vertically until their boxes stop overlapping.
    fig = ax.figure
    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()
    px_to_pt = 72 / fig.dpi
    for _ in range(max_iter):
        moved = False
        boxes = [t.get_window_extent(renderer) for t in texts]
        for i in range(len(texts)):
            for j in range(i + 1, len(texts)):
                a, b = boxes[i], boxes[j]
                if not a.overlaps(b):
                    continue
                shift = (min(a.y1, b.y1) - max(a.y0, b.y0)) / 2 + pad_px
                upper, lower = (i, j) if (a.y0 + a.y1) >= (b.y0 + b.y1) else (j, i)
                for k, sign in ((upper, 1), (lower, -1)):
                    x, y = texts[k].xyann
                    texts[k].xyann = (x, y + sign * shift * px_to_pt)
                    boxes[k] = texts[k].get_window_extent(renderer)
                moved = True
        if not moved:
            break


LEADER = dict(arrowstyle="-", color="#bbbbbb", linewidth=0.6, shrinkA=0, shrinkB=4)


def fmt_minutes(m):
    if not np.isfinite(m):
        return "n/a"
    if m < 10:
        return f"{m:.1f} min"
    if m < 120:
        return f"{m:.0f} min"
    return f"{m / 60:.1f} h"

## Load the gap series

In [ ]:
def find_results_dir(extra_roots=()):
    # Returns the first folder containing summary.csv, or None.
    candidates = [Path("results"), Path("../results"), Path("out"), Path("../out"),
                  Path("/content/results"), Path("/content/out"), Path("/content/mini-project-mtp/results")]
    for c in candidates:
        if (c / "summary.csv").exists():
            return c
    for root in [*map(Path, extra_roots), Path("."), Path("/content")]:
        if not root.exists():
            continue
        for pattern in ("*/summary.csv", "*/*/summary.csv", "*/*/*/summary.csv"):
            hits = sorted(p for p in root.glob(pattern) if "fit_out" not in p.parts)
            if hits:
                return hits[0].parent
    return None


RESULTS_DIR = find_results_dir()
if RESULTS_DIR is None:
    try:
        from google.colab import files
    except ImportError:
        files = None
    if files is None:
        raise FileNotFoundError("Could not find a results/ folder with summary.csv. Put this notebook next to "
                                "the repo's results/ folder, or run pull_all_events.ipynb first.")
    print("No results folder found. Upload results.zip (a zip of the repo's results/ folder).")
    uploaded = files.upload()
    for name_ in uploaded:
        if name_.lower().endswith(".zip"):
            with zipfile.ZipFile(name_) as z:
                z.extractall("uploaded_results")
    RESULTS_DIR = find_results_dir(extra_roots=["uploaded_results"])
    if RESULTS_DIR is None:
        raise FileNotFoundError("The upload did not contain summary.csv. Zip the whole results/ folder and try again.")
print("Reading data from:", RESULTS_DIR.resolve())

In [ ]:
summary = pd.read_csv(RESULTS_DIR / "summary.csv")
if "bar_minutes" not in summary.columns or (summary["bar_minutes"] != 1).any():
    raise ValueError("These are not 1-minute results. Re-run the fixed pull_all_events.ipynb (BAR_MINUTES = 1).")
not_ok = summary[summary["status"] != "ok"]
if not not_ok.empty:
    print("Skipping events whose pull did not finish cleanly:")
    print(not_ok[["label", "status"]].to_string(index=False))
summary = summary[summary["status"] == "ok"].reset_index(drop=True)

EV = {}
for _, row in summary.iterrows():
    df = pd.read_csv(RESULTS_DIR / row["label"] / "gap.csv")
    df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True)
    df = df.sort_values("timestamp").drop_duplicates("timestamp").reset_index(drop=True)
    df["is_outlier"] = df["is_outlier"].astype(str).str.lower().eq("true")
    df["kalshi_volume"] = pd.to_numeric(df["kalshi_volume"], errors="coerce").fillna(0.0)
    # Each row's minute number from the start. The pull notebook puts every row on an exact minute;
    # anything else would break the exact-step pairing, so check it.
    secs = (df["timestamp"] - df["timestamp"].iloc[0]).dt.total_seconds().to_numpy()
    if not np.allclose(secs % 60, 0):
        raise ValueError(f"{row['label']}: timestamps are not on whole minutes")
    df["minute"] = (secs // 60).astype(int)
    EV[row["label"]] = df

CATEGORY = dict(zip(summary["label"], summary["category"]))
LABELS = [l for c in CATEGORY_ORDER for l in summary.loc[summary["category"] == c, "label"]]
name = lambda label: SHORT_NAMES.get(label, label)
print(f"Loaded {len(EV)} markets:")
for label in LABELS:
    print(f"  {CATEGORY[label]:9s} {name(label):20s} {len(EV[label]):6d} rows")

## The fitting machinery

All fits below use these functions. `make_pairs` builds (G(t), G(t+Δ) − G(t))
for rows exactly Δ minutes apart; `fit_threshold` tries every c on the grid and
keeps the best; the bootstrap refits both c and κ on resampled hours.

In [ ]:
def centre_values(minute, gap, how=CENTRE, train_until=None):
    # The level each gap is measured from (see "Choices and why").
    #   rolling24h: median of the gap over the previous 24 hours (only past data; the first hour, before
    #               there are 60 observations, uses the median of everything so far)
    #   median:     one median for the whole period (or for minutes <= train_until)
    #   none:       zero
    if how == "none":
        return np.zeros(len(gap))
    if how == "median":
        src = gap if train_until is None else gap[minute <= train_until]
        return np.full(len(gap), float(np.median(src)))
    if how == "rolling24h":
        s = pd.Series(gap, index=pd.to_datetime(minute, unit="m"))
        roll = s.rolling("24h", min_periods=60).median()
        return roll.fillna(s.expanding().median()).to_numpy()
    raise ValueError(how)


def centre_series(df, how=CENTRE, train_until=None):
    return centre_values(df["minute"].to_numpy(), df["gap"].to_numpy(), how, train_until)


def make_pairs(minute, dev, flagged, step):
    # Rows exactly `step` minutes apart, neither flagged. Returns start minute, gap, change.
    pos = np.full(minute[-1] + step + 1, -1)
    pos[minute] = np.arange(len(minute))
    j = pos[minute + step]
    ok = j >= 0
    ok &= ~flagged & ~flagged[np.where(ok, j, 0)]
    i = np.flatnonzero(ok)
    j = j[ok]
    return {"minute": minute[i], "g0": dev[i], "dg": dev[j] - dev[i]}


def event_pairs(label, step=STEP, centre=CENTRE, keep_flagged=False):
    df = EV[label]
    dev = df["gap"].to_numpy() - centre_series(df, centre)
    flagged = np.zeros(len(df), bool) if keep_flagged else df["is_outlier"].to_numpy()
    return make_pairs(df["minute"].to_numpy(), dev, flagged, step)


def excess(g, c):
    return np.sign(g) * np.maximum(np.abs(g) - c, 0.0)


def block_stats(p, c_grid=C_GRID):
    # Per one-hour block and per candidate c: the sums the least-squares fit needs.
    block = p["minute"] // BLOCK_MINUTES
    starts = np.flatnonzero(np.r_[True, block[1:] != block[:-1]])
    X = excess(p["g0"][:, None], c_grid[None, :])
    dg = p["dg"]
    return {"sxy": np.add.reduceat(X * dg[:, None], starts, axis=0),
            "sxx": np.add.reduceat(X * X, starts, axis=0),
            "cnt": np.add.reduceat((X != 0).astype(float), starts, axis=0),
            "syy": np.add.reduceat(dg * dg, starts)}


def fit_from_weights(st, w, c_grid=C_GRID):
    # w: block weights, shape (n_blocks,) or (n_draws, n_blocks).
    Sxy, Sxx, N, Syy = w @ st["sxy"], w @ st["sxx"], w @ st["cnt"], w @ st["syy"]
    explained = np.where(Sxx > 0, Sxy ** 2 / np.where(Sxx > 0, Sxx, 1.0), 0.0)
    sse = np.where(N >= MIN_BEYOND, Syy[..., None] - explained, np.inf)
    i = np.argmin(sse, axis=-1)
    take = lambda A: np.take_along_axis(A, np.asarray(i)[..., None], -1)[..., 0]
    with np.errstate(invalid="ignore", divide="ignore"):
        kappa = -take(Sxy) / take(Sxx)
        r2 = np.where(N >= MIN_BEYOND, explained / Syy[..., None], np.nan)
    return c_grid[i], kappa, r2, take(N)


def bootstrap_weights(n_blocks, n_draws, rng):
    return np.stack([np.bincount(rng.integers(0, n_blocks, n_blocks), minlength=n_blocks)
                     for _ in range(n_draws)]).astype(float)


def fit_threshold(p, n_boot=0, rng=None, c_grid=C_GRID):
    st = block_stats(p, c_grid)
    nb = len(st["syy"])
    c, k, r2, nbeyond = fit_from_weights(st, np.ones(nb), c_grid)
    out = {"c": float(c), "kappa": float(k), "r2_curve": r2, "n_beyond": int(nbeyond), "n_pairs": len(p["dg"])}
    if n_boot:
        cb, kb, _, _ = fit_from_weights(st, bootstrap_weights(nb, n_boot, rng), c_grid)
        out["c_boot"], out["kappa_boot"] = cb, kb
    return out


def fit_c0(p):
    g, dg = p["g0"], p["dg"]
    return -np.sum(g * dg) / np.sum(g * g)


def half_life(kappa, step=STEP):
    # Minutes for the excess beyond c to halve. κ >= 1 (all of it gone within one step) is
    # clipped to 0.999 so it still plots; κ <= 0 (no pull) gives NaN.
    k = np.minimum(np.asarray(kappa, dtype=float), 0.999)
    with np.errstate(divide="ignore", invalid="ignore"):
        hl = step * np.log(0.5) / np.log(1 - k)
    return np.where(k > 0, hl, np.nan)

# Part A — Results

## Figure 1 — the model in one picture

In [ ]:
c_ex, k_ex = 2.0, 0.3
x = np.linspace(-6, 6, 400)
y = np.where(np.abs(x) > c_ex, -k_ex * (x - np.sign(x) * c_ex), 0.0)
fig, ax = plt.subplots(figsize=(10, 5.2))
ax.axvspan(-c_ex, c_ex, color=BAND, zorder=0)
ax.plot(x, y, color=INK, linewidth=2.5)
ax.axhline(0, color="#999999", linewidth=0.8)
ax.axvline(0, color="#999999", linewidth=0.8)
ax.text(0, 1.05, "NO-TRADE BAND\nthe gap is smaller than the cost\nof trading it away, so nothing\npulls it back: it just drifts",
        ha="center", va="bottom", fontsize=9.5, color=INK)
ax.annotate("", xy=(-c_ex, -1.55), xytext=(c_ex, -1.55), arrowprops=dict(arrowstyle="<->", color=INK, linewidth=1.2))
ax.text(0, -1.45, "width = 2c  (c = implied trading cost)", ha="center", va="bottom", fontsize=9.5)
ax.text(4.1, 0.25, "Kalshi above its usual level vs\nPolymarket by more than c: traders\nbuy the cheap side and sell the\nexpensive one, pulling the gap back",
        ha="center", va="bottom", fontsize=9.5, color=INK)
ax.text(-4.1, -0.25, "the other direction: the gap is\npushed back up toward the band", ha="center", va="top",
        fontsize=9.5, color=INK)
ax.annotate("steepness = κ\n(share of the excess gap\nthat closes per step)", xy=(4.6, -k_ex * 2.6), xytext=(2.3, -1.25),
            fontsize=9.5, arrowprops=dict(arrowstyle="->", color=MUTED))
ax.set_xlim(-6, 6)
ax.set_ylim(-1.8, 1.8)
ax.set_xlabel("Gap now, measured from the market's usual gap (cents)")
ax.set_ylabel("Expected change in the gap\nover the next step (cents)")
ax.set_title(f"The threshold model (illustration with c = {c_ex:.0f}¢, κ = {k_ex})")
caption(fig, "How to read this: the horizontal axis is how far the price gap between the two platforms is from its "
             "usual level (its median) right now; the vertical axis is how much it is expected to change over the next step. Inside "
             "the grey band the line is flat: small gaps are not worth trading, so on average they don't shrink. Outside "
             "the band the line slopes down: big gaps are pulled back toward the band. The notebook estimates the band "
             "half-width c and the steepness κ for nine real markets, and reports κ as a half-life in minutes.")
fig.tight_layout()
fig.savefig(OUT / "fig01_model.png", dpi=150, bbox_inches="tight")
plt.show()

## Main fit: 5-minute step, gaps measured from each market's median gap

In [ ]:
PAIRS, FITS, rows = {}, {}, []
for label in LABELS:
    p = event_pairs(label)
    PAIRS[label] = p
    f = fit_threshold(p, N_BOOT, rng)
    FITS[label] = f
    hl_boot = half_life(f["kappa_boot"])
    rows.append({
        "label": label, "event": name(label), "category": CATEGORY[label],
        "n_pairs": f["n_pairs"], "pairs_beyond_c": f["n_beyond"],
        "median_gap_cents": 100 * float(EV[label]["gap"].median()),
        "c_cents": 100 * f["c"],
        "c_lo_cents": 100 * np.percentile(f["c_boot"], 2.5), "c_hi_cents": 100 * np.percentile(f["c_boot"], 97.5),
        "kappa_5min": f["kappa"],
        "kappa_lo": np.nanpercentile(f["kappa_boot"], 2.5), "kappa_hi": np.nanpercentile(f["kappa_boot"], 97.5),
        "half_life_min": float(half_life(f["kappa"])),
        "hl_lo": np.nanpercentile(hl_boot, 2.5), "hl_hi": np.nanpercentile(hl_boot, 97.5),
        "r2_threshold": float(np.nanmax(f["r2_curve"])), "r2_c0": float(f["r2_curve"][0]),
        "kappa_c0_model": fit_c0(p),
    })
fits = pd.DataFrame(rows)

act = []
for label in LABELS:
    df = EV[label]
    span_min = df["minute"].iloc[-1] + 1
    pk = df["kalshi"]
    act.append({
        "label": label,
        "kalshi_cost_cents": 100 * (df["kalshi_spread"] / 2 + KALSHI_FEE_RATE * pk * (1 - pk)).mean(),
        "kalshi_contracts_per_sec": df["kalshi_volume"].sum() / (span_min * 60),
        "kalshi_trade_minute_share": (df["kalshi_volume"] > 0).sum() / span_min,
        "kalshi_spread_cents": 100 * df["kalshi_spread"].mean(),
        "poly_move_share": float((df["polymarket"].diff() != 0).iloc[1:].mean()),
    })
results = fits.merge(pd.DataFrame(act), on="label")
R = results.set_index("label")
results[["event", "category", "n_pairs", "median_gap_cents", "c_cents", "c_lo_cents", "c_hi_cents",
         "half_life_min", "hl_lo", "hl_hi", "kalshi_cost_cents"]].round(2)

## Figure 2 — the model against the data

In [ ]:
def binned(p, width=0.005, min_count=30):
    b = np.round(p["g0"] / width) * width
    g = pd.DataFrame({"b": b, "dg": p["dg"]}).groupby("b")["dg"].agg(["mean", "count"])
    return g[g["count"] >= min_count]


fig, axes = plt.subplots(3, 3, figsize=(13, 11))
for ax, label in zip(axes.flat, LABELS):
    cat = CATEGORY[label]
    r = R.loc[label]
    b = binned(PAIRS[label])
    xb, yb = 100 * b.index.to_numpy(), 100 * b["mean"].to_numpy()
    c, k = r["c_cents"], r["kappa_5min"]
    lo, hi = min(xb.min(), -c - 1), max(xb.max(), c + 1)
    xs = np.linspace(lo, hi, 400)
    ax.axvspan(-c, c, color=BAND, zorder=0)
    ax.plot(xs, -r["kappa_c0_model"] * xs, color="#999999", linestyle="--", linewidth=1.3, label="no band (c = 0)")
    ax.plot(xs, -k * excess(xs, c), color=INK, linewidth=2,
            label=f"band c = {c:.1f}¢, half-life {fmt_minutes(r['half_life_min'])}")
    ax.scatter(xb, yb, s=18 + 140 * b["count"] / b["count"].max(), color=CATEGORY_COLORS[cat],
               marker=CATEGORY_MARKERS[cat], edgecolor="white", linewidth=0.8, zorder=3)
    pad = 0.15 * (yb.max() - yb.min() + 1e-9)
    ax.set_ylim(yb.min() - pad, yb.max() + pad)
    ax.set_xlim(lo, hi)
    ax.axhline(0, color="#aaaaaa", linewidth=0.7)
    ax.set_title(f"{name(label)} ({cat})", fontsize=10)
    ax.legend(loc="upper right", fontsize=7.5, frameon=True, framealpha=0.85, edgecolor="none")
for ax in axes[-1]:
    ax.set_xlabel("Gap now, from its usual level (cents)")
for ax in axes[:, 0]:
    ax.set_ylabel("Avg change over\nnext 5 minutes (cents)")
fig.suptitle("Does the gap get pulled back? Data (markers) vs the fitted model (black line)", fontsize=13, fontweight="bold")
caption(fig, "How to read this: each panel is one market. Markers group the 5-minute steps by how far the gap was from "
             "its usual level (the market's median gap; bigger marker = more data) and show the average change over the next 5 "
             "minutes. Markers above zero on the left and below zero on the right mean gaps are pulled back. The grey "
             "band is the fitted no-trade band (−c to +c) and the black line the fitted model; the dashed line is the "
             "simpler model with no band. Where the markers are flat in the middle and fall away at the edges, the band "
             "describes the data better than the dashed line.", width=165)
fig.tight_layout()
fig.savefig(OUT / "fig02_model_vs_data.png", dpi=150, bbox_inches="tight")
plt.show()

## Figure 3 — does the answer depend on the time step? (validation check 1)

If gaps really close at the fitted speed, the band c and the half-life in
minutes should come out about the same whether we look 1, 5 or 15 minutes
ahead. If the 1-minute half-life is much shorter than the others, part of what
looks like fast closing is quotes flickering and snapping back within a minute.

In [ ]:
step_rows = []
for label in LABELS:
    for s in STEPS_TO_COMPARE:
        f = FITS[label] if s == STEP else fit_threshold(event_pairs(label, step=s))
        step_rows.append({"label": label, "event": name(label), "step": s, "c_cents": 100 * f["c"],
                          "kappa": f["kappa"], "half_life_min": float(half_life(f["kappa"], s))})
steps_df = pd.DataFrame(step_rows)
steps_df.to_csv(OUT / "step_check.csv", index=False)

order = results.sort_values("c_cents").reset_index(drop=True)
ypos = {l: i for i, l in enumerate(order["label"])}
style = {1: dict(marker="o", facecolor="white", edgecolor=MUTED, s=55, label="1-minute step"),
         5: dict(marker="o", facecolor=INK, edgecolor=INK, s=60, label="5-minute step (main)"),
         15: dict(marker="^", facecolor="#9a9a9a", edgecolor="#9a9a9a", s=55, label="15-minute step")}
off = {1: -0.2, 5: 0.0, 15: 0.2}
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6), sharey=True)
for _, r in steps_df.iterrows():
    y0 = ypos[r["label"]] + off[r["step"]]
    st = style[r["step"]]
    ax1.scatter(r["c_cents"], y0, marker=st["marker"], facecolor=st["facecolor"], edgecolor=st["edgecolor"], s=st["s"], zorder=3)
    if np.isfinite(r["half_life_min"]):
        ax2.scatter(r["half_life_min"], y0, marker=st["marker"], facecolor=st["facecolor"], edgecolor=st["edgecolor"], s=st["s"], zorder=3)
ax1.set_yticks(range(len(order)), order["event"])
ax1.set_xlim(left=0)
ax1.set_xlabel("c: no-trade band (cents)")
ax1.set_title("Band c fitted at each step")
plain_log(ax2, "x")
ax2.set_xlabel("Half-life of the excess gap (minutes, log scale)")
ax2.set_title("Half-life fitted at each step")
handles = [plt.Line2D([], [], linestyle="", marker=v["marker"], markerfacecolor=v["facecolor"], markeredgecolor=v["edgecolor"],
                      markersize=8, label=v["label"]) for v in style.values()]
fig.legend(handles=handles, loc="upper center", ncol=3, frameon=False, bbox_to_anchor=(0.5, 1.02))
fig.suptitle("Check 1: do 1-, 5- and 15-minute steps give the same answer?", fontsize=13, fontweight="bold", y=1.07)
caption(fig, "How to read this: each row is a market and each marker a fit with a different step (how far ahead we look). "
             "A real gap-closing process gives about the same band and the same half-life in minutes at every step, so "
             "the three markers should sit together. When the white 1-minute marker sits far to the left of the others "
             "on the right panel, the 1-minute fit is picking up quotes that flicker and snap back within a minute, not "
             "real closing — which is why the main fit uses 5 minutes. When the half-life keeps growing from 5 to 15 "
             "minutes as well, the gap closes in two speeds — a fast part and a slower leftover — which one κ can't "
             "describe; the reported half-life then describes the first few minutes.", width=170)
fig.tight_layout()
fig.savefig(OUT / "fig03_step_check.png", dpi=150, bbox_inches="tight")
plt.show()

steps_df.pivot(index="event", columns="step", values="half_life_min").round(1).rename(columns=lambda s: f"half-life @ {s} min")

## Figure 4 — the band c and the half-life for each market

In [ ]:
order = results.sort_values("c_cents").reset_index(drop=True)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6), sharey=True)
for i, r in order.iterrows():
    color, marker = CATEGORY_COLORS[r["category"]], CATEGORY_MARKERS[r["category"]]
    ax1.plot([r["c_lo_cents"], r["c_hi_cents"]], [i, i], color=color, linewidth=2, solid_capstyle="round")
    ax1.scatter(r["c_cents"], i, color=color, marker=marker, s=70, edgecolor="white", linewidth=1.5, zorder=3)
    ax1.scatter(r["kalshi_cost_cents"], i, marker="|", color=INK, s=220, linewidth=2, zorder=4)
    ax2.plot([r["hl_lo"], r["hl_hi"]], [i, i], color=color, linewidth=2, solid_capstyle="round")
    ax2.scatter(r["half_life_min"], i, color=color, marker=marker, s=70, edgecolor="white", linewidth=1.5, zorder=3)
    ax2.annotate(fmt_minutes(r["half_life_min"]), (r["hl_hi"], i), xytext=(6, 0), textcoords="offset points",
                 va="center", fontsize=8.5, color=MUTED)
ax1.set_yticks(range(len(order)), order["event"])
ax1.set_xlim(left=0)
ax1.set_xlabel("c: half-width of the no-trade band (cents)")
ax1.set_title("How big a gap must be before it closes")
plain_log(ax2, "x")
ax2.set_xlim(order["hl_lo"].min() / 1.5, order["hl_hi"].max() * 3)
ax2.set_xlabel("Half-life of the excess beyond c (minutes, log scale)")
ax2.set_title("How fast the excess closes")
cost_handle = plt.Line2D([], [], color=INK, marker="|", linestyle="", markersize=14, markeredgewidth=2,
                         label="Kalshi-side cost of one trade (half spread + fee)")
fig.legend(handles=category_handles([cost_handle]), loc="upper center", ncol=4, frameon=False, bbox_to_anchor=(0.5, 1.03))
fig.suptitle("Fitted no-trade band c and half-life, with 95% bootstrap ranges (5-minute step)", fontsize=13,
             fontweight="bold", y=1.08)
caption(fig, "How to read this: each row is one market; markers are the estimates and bars their 95% ranges. Left: the "
             "band c — gaps within c of their usual level (the market's median gap) are not traded away. The black tick is "
             "what one trade costs on the Kalshi side alone (half the bid–ask spread plus Kalshi's fee); the full cost of "
             "closing a gap is higher, since the Polymarket side costs extra. Right: once a gap is wider than c, how many "
             "minutes it takes for the excess to halve (shorter = faster). Wide bars mean the data can't pin the number down; "
             "a bar reaching 0.5 min means some resamples had the whole excess vanish within one 5-minute step, the fastest "
             "this step size can measure.",
        width=170)
fig.tight_layout()
fig.savefig(OUT / "fig04_c_and_half_life.png", dpi=150, bbox_inches="tight")
plt.show()

## Figure 5 — is c really a transaction cost?

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 7))
lim = max(results["c_cents"].max(), results["kalshi_cost_cents"].max()) * 1.35
ax.plot([0, lim], [0, lim], color="#999999", linestyle="--", linewidth=1)
ax.text(lim * 0.97, lim * 0.93, "c = Kalshi-side cost", ha="right", va="top", fontsize=9, color=MUTED)
texts = []
for _, r in results.iterrows():
    color, marker = CATEGORY_COLORS[r["category"]], CATEGORY_MARKERS[r["category"]]
    top = min(r["c_hi_cents"], lim * 0.98)
    ax.plot([r["kalshi_cost_cents"]] * 2, [r["c_lo_cents"], top], color=color, linewidth=1.5, alpha=0.55)
    if r["c_hi_cents"] > lim * 0.98:
        ax.plot(r["kalshi_cost_cents"], top, marker="^", color=color, markersize=5, alpha=0.7)
        ax.annotate(f"range continues\nup to {r['c_hi_cents']:.1f}¢", (r["kalshi_cost_cents"], top), xytext=(-8, -2),
                    textcoords="offset points", ha="right", va="top", fontsize=7.5, color=MUTED)
    ax.scatter(r["kalshi_cost_cents"], r["c_cents"], color=color, marker=marker, s=80, edgecolor="white", linewidth=1.5, zorder=3)
    texts.append(ax.annotate(r["event"], (r["kalshi_cost_cents"], r["c_cents"]), xytext=(10, 0), textcoords="offset points",
                             va="center", fontsize=8.5, color="#333333", arrowprops=LEADER,
                             bbox=dict(boxstyle="round,pad=0.15", fc="white", ec="none", alpha=0.8)))
ax.set_xlim(0, lim)
ax.set_ylim(0, lim)
ax.set_xlabel("Kalshi-side cost of one trade (cents)\n(half the bid–ask spread + Kalshi fee)")
ax.set_ylabel("Fitted band half-width c (cents)")
ax.set_title("Fitted band c vs what trading costs on Kalshi")
ax.legend(handles=category_handles(), loc="upper left", frameon=False)
caption(fig, "How to read this: if c is a transaction cost it should be at least the cost we can measure. Each marker is "
             "one market (vertical bar = 95% range of c). Markers near the dashed line have a band about equal to the "
             "Kalshi-side cost; markers above it need a bigger gap before it closes (the Polymarket side and moving money "
             "between platforms add cost). Markers well below the line have gaps closing at less than Kalshi's quoted "
             "cost, which suggests c there is picking up something other than trading cost.", width=105, y=-0.03)
fig.tight_layout()
dodge_labels(ax, texts)
fig.savefig(OUT / "fig05_c_vs_cost.png", dpi=150, bbox_inches="tight")
plt.show()

# Part B — Validation

## Check 2 — does the method recover known values? (simulation)

For each market we simulate **fake gap series where we know the true c and κ**
(set to that market's fitted values), and run exactly the same fitting code on
them. The fake data copies the real data's problems:

* same timeline, with the same missing minutes;
* prices rounded to half-cent steps;
* short-lived quote **flicker** (noise that reverts the next minute), sized from
  how much the real 1-minute changes bounce back;
* **stale** minutes where the quote doesn't update;
* two settings are tuned so the fake data look like the real data: the noise
  size (so 5-minute changes are as variable as the real ones) and the stale
  probability (so the share of minutes with no change matches the real share,
  counting the zeros that rounding already produces).

The fake data are then fitted exactly like the real data, including measuring
the gap from its median. The true process runs minute by
minute with per-minute pull `κ₁ = 1 − (1 − κ₅)^(1/5)`, so the 5-minute fit has to
cope with the same time aggregation as on real data. If the fit is sound, the
estimates scatter around the true values and the 95% ranges contain the truth
about 95% of the time. The fake data have no drifting offset and no news jumps,
so passing this check is necessary, not sufficient.

In [ ]:
def simulate(minute, c, k1, sigma, tau, stale, n_rep, rng, tick=TICK, burn=500):
    # Minute-by-minute threshold process, observed with flicker, rounding and stale quotes,
    # kept only at the minutes present in `minute`. Returns array (len(minute), n_rep).
    n_total = minute[-1] + 1
    keep = np.zeros(n_total, bool)
    keep[minute] = True
    out = np.empty((len(minute), n_rep))
    L = np.zeros(n_rep)
    prev = np.zeros(n_rep)
    row = 0
    for t in range(-burn, n_total):
        L = L - k1 * excess(L, c) + sigma * rng.standard_normal(n_rep)
        obs = L + tau * rng.standard_normal(n_rep) if tau > 0 else L.copy()
        if tick:
            obs = np.round(obs / tick) * tick
        if stale > 0:
            obs = np.where(rng.random(n_rep) < stale, prev, obs)
        prev = obs
        if t >= 0 and keep[t]:
            out[row] = obs
            row += 1
    return out


def sim_pairs(minute, obs_col, step, centre=CENTRE):
    # Same treatment as the real data: measure from the centre, exact-step pairs.
    dev = obs_col - centre_values(minute, obs_col, centre)
    return make_pairs(minute, dev, np.zeros(len(minute), bool), step)


def noise_profile(label):
    # Real-data targets: flicker size (from how much 1-minute changes reverse), share of unchanged
    # minutes, and the variance of 5-minute changes.
    df = EV[label]
    minute = df["minute"].to_numpy()
    dev = df["gap"].to_numpy() - centre_series(df)
    p1 = make_pairs(minute, dev, df["is_outlier"].to_numpy(), 1)
    pos = np.full(minute[-1] + 2, -1)
    pos[p1["minute"]] = np.arange(len(p1["minute"]))
    nxt = pos[np.minimum(p1["minute"] + 1, minute[-1] + 1)]
    ok = nxt >= 0
    a, b = p1["dg"][ok], p1["dg"][nxt[ok]]
    tau = np.sqrt(max(-np.mean((a - a.mean()) * (b - b.mean())), 0.0))
    zero_share = float(np.mean(np.abs(p1["dg"]) < 1e-9))
    var5 = float(np.var(event_pairs(label)["dg"]))
    return tau, zero_share, var5


def calibrate(minute, c, k1, tau, zero_share, var5, rng, n_iter=4, n_rep=10):
    # Tunes the noise size sigma and the stale probability so the fake data match the real
    # variance of 5-minute changes and the real share of unchanged minutes.
    sigma, stale = np.sqrt(var5 / STEP), 0.0
    for _ in range(n_iter):
        trial = simulate(minute, c, k1, sigma, tau, stale, n_rep, rng)
        v5 = np.mean([np.var(sim_pairs(minute, trial[:, j], STEP)["dg"]) for j in range(n_rep)])
        z = np.mean([np.mean(np.abs(sim_pairs(minute, trial[:, j], 1)["dg"]) < 1e-9) for j in range(n_rep)])
        z_round = (z - stale) / (1 - stale)              # zeros that rounding alone produces
        stale = float(np.clip((zero_share - z_round) / (1 - z_round), 0.0, 0.98))
        sigma *= np.sqrt(var5 / v5)
    return sigma, stale


def sim_fit(minute, obs, step=STEP, n_boot=0, rng=None, c_grid=C_GRID, centre=CENTRE):
    return [fit_threshold(sim_pairs(minute, obs[:, r], step, centre), n_boot, rng, c_grid) for r in range(obs.shape[1])]


SIM = {}
t0 = time.time()
for label in LABELS:
    r = R.loc[label]
    minute = EV[label]["minute"].to_numpy()
    c_true, k5_true = r["c_cents"] / 100, min(r["kappa_5min"], 0.999)
    k1 = 1 - (1 - k5_true) ** (1 / STEP)
    tau, zero_share, var5 = noise_profile(label)
    sigma, stale = calibrate(minute, c_true, k1, tau, zero_share, var5, rng)
    obs = simulate(minute, c_true, k1, sigma, tau, stale, N_SIM, rng)
    fits_sim = sim_fit(minute, obs, n_boot=N_BOOT_SIM, rng=rng)
    c_hat = np.array([f["c"] for f in fits_sim])
    k_hat = np.array([f["kappa"] for f in fits_sim])
    c_cov = np.mean([np.percentile(f["c_boot"], 2.5) <= c_true + 1e-12 <= np.percentile(f["c_boot"], 97.5) + 2e-12
                     for f in fits_sim])
    hl_cov = np.mean([np.nanpercentile(half_life(f["kappa_boot"]), 2.5) <= half_life(k5_true)
                      <= np.nanpercentile(half_life(f["kappa_boot"]), 97.5) for f in fits_sim])
    SIM[label] = dict(c_true=c_true, hl_true=float(half_life(k5_true)), c_hat=c_hat, hl_hat=half_life(k_hat),
                      c_cov=c_cov, hl_cov=hl_cov, tau=tau, stale=stale, zero_share=zero_share, sigma=sigma, k1=k1)
    print(f"  {name(label):20s} simulated and refitted {N_SIM} series  ({time.time() - t0:.0f}s)")

sim_df = pd.DataFrame([{
    "event": name(l), "c_true_cents": 100 * s["c_true"], "c_median_est": 100 * np.median(s["c_hat"]),
    "c_coverage_95": s["c_cov"], "half_life_true": s["hl_true"], "half_life_median_est": np.nanmedian(s["hl_hat"]),
    "half_life_coverage_95": s["hl_cov"], "flicker_cents": 100 * s["tau"], "real_unchanged_share": s["zero_share"],
    "stale_prob_used": s["stale"]} for l, s in SIM.items()])
sim_df.to_csv(OUT / "simulation_recovery.csv", index=False)
sim_df.round(3)

### Figure 6 — simulation: estimates vs the truth

In [ ]:
order = results.sort_values("c_cents").reset_index(drop=True)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6.5), sharey=True)
jit = np.random.default_rng(0)
for i, r in order.iterrows():
    s = SIM[r["label"]]
    color, marker = CATEGORY_COLORS[r["category"]], CATEGORY_MARKERS[r["category"]]
    yj = i + jit.uniform(-0.22, 0.22, len(s["c_hat"]))
    ax1.scatter(100 * s["c_hat"], yj, s=9, color=color, alpha=0.45, linewidth=0)
    ax1.scatter(100 * s["c_true"], i, marker="|", s=380, color=INK, linewidth=2.5, zorder=4)
    ax1.annotate(f"{100 * s['c_cov']:.0f}%", (1, i), xycoords=("axes fraction", "data"), xytext=(4, 0),
                 textcoords="offset points", va="center", fontsize=8.5, color=MUTED)
    hl = s["hl_hat"]
    ax2.scatter(hl[np.isfinite(hl)], yj[np.isfinite(hl)], s=9, color=color, alpha=0.45, linewidth=0)
    ax2.scatter(s["hl_true"], i, marker="|", s=380, color=INK, linewidth=2.5, zorder=4)
    ax2.annotate(f"{100 * s['hl_cov']:.0f}%", (1, i), xycoords=("axes fraction", "data"), xytext=(4, 0),
                 textcoords="offset points", va="center", fontsize=8.5, color=MUTED)
ax1.set_yticks(range(len(order)), order["event"])
ax1.set_xlim(left=0)
ax1.set_xlabel("Band c estimated from simulated data (cents)")
ax1.set_title("Band c: estimates (dots) vs truth (black tick)")
plain_log(ax2, "x")
ax2.set_xlabel("Half-life estimated from simulated data (minutes, log scale)")
ax2.set_title("Half-life: estimates (dots) vs truth (black tick)")
fig.suptitle(f"Check 2: fitting {N_SIM} simulated series per market where the true values are known",
             fontsize=13, fontweight="bold", y=1.02)
caption(fig, "How to read this: for each market we made fake data with a known band and half-life (black ticks, set to "
             "that market's fitted values) and the same gaps, rounding, flicker and stale quotes as the real data, then "
             f"fitted it {N_SIM} times. Each dot is one fit. If the method works, the dots cluster around the black tick. "
             "The percentage at the right is how often the 95% range from the bootstrap contained the true value — it "
             "should be close to 95%. Dots spread far from the tick, or a percentage well below 95%, mean the method "
             "can't recover that market's values reliably, and its real-data result should not be trusted.", width=170)
fig.tight_layout()
fig.savefig(OUT / "fig06_simulation_recovery.png", dpi=150, bbox_inches="tight")
plt.show()

### Figure 7 — simulation: can the method tell "no band" from "a band"?

Same idea, with a fixed template (the timeline and noise of the market with the
most data) and bands set to exactly 0, 1, 2 and 3¢. The fake series are fitted
twice: with the main choice (gap measured from its median) and with a trailing
24-hour median instead, to show why the main fit doesn't use a trailing median.

In [ ]:
template = results.sort_values("n_pairs").iloc[-1]["label"]
minute_t = EV[template]["minute"].to_numpy()
k5_known = 0.3
k1_t = 1 - (1 - k5_known) ** (1 / STEP)
tau_t, zero_t, var5_t = noise_profile(template)
sigma_t, stale_t = calibrate(minute_t, 0.01, k1_t, tau_t, zero_t, var5_t, rng)
KNOWN_CENTRES = {"median": "Gap from its median (main)", "rolling24h": "Gap from a trailing 24-hour median"}
known = {cn: {} for cn in KNOWN_CENTRES}
for c_c in KNOWN_C_CENTS:
    obs = simulate(minute_t, c_c / 100, k1_t, sigma_t, tau_t, stale_t, N_SIM_KNOWN, rng)
    for cn in KNOWN_CENTRES:
        known[cn][c_c] = 100 * np.array([f["c"] for f in sim_fit(minute_t, obs, centre=cn)])

fig, axes = plt.subplots(1, 2, figsize=(13, 5.8), sharey=True)
mx = max(KNOWN_C_CENTS) + 1.5
for ax, (cn, title) in zip(axes, KNOWN_CENTRES.items()):
    ax.plot([0, mx], [0, mx], color="#999999", linestyle="--", linewidth=1, label="estimate = truth")
    for c_c, est in known[cn].items():
        ax.scatter(c_c + np.random.default_rng(c_c).uniform(-0.12, 0.12, len(est)), est, s=14,
                   color=CATEGORY_COLORS["Sports"], alpha=0.5, linewidth=0)
        ax.plot([c_c - 0.25, c_c + 0.25], [np.median(est)] * 2, color=INK, linewidth=2.5)
    ax.plot([], [], color=INK, linewidth=2.5, label="median estimate")
    ax.set_xticks(list(KNOWN_C_CENTS), [f"{c}¢" for c in KNOWN_C_CENTS])
    ax.set_xlim(-0.6, max(KNOWN_C_CENTS) + 0.6)
    ax.set_ylim(0, mx)
    ax.set_xlabel("True band c used to make the fake data")
    ax.set_title(title)
    ax.legend(loc="upper left", frameon=False)
axes[0].set_ylabel("Band c estimated from it (cents)")
fig.suptitle(f"Check 2b: recovering known bands (template: {name(template)}, true half-life "
             f"{fmt_minutes(float(half_life(k5_known)))})", fontsize=13, fontweight="bold", y=1.02)
within = {c: 100 * np.mean(np.abs(e - c) <= 0.5) for c, e in known["median"].items()}
caption(fig, f"How to read this: we made {N_SIM_KNOWN} fake series for each true band (0, 1, 2 and 3¢), with the timeline, "
             "rounding, flicker and stale quotes of the real market with the most data, and fitted each one. Each dot is "
             "one estimate; the black bar is their median; dots on the dashed line mean the band was recovered exactly. "
             "Left, the main method: the share of fits within 0.5¢ of the truth is "
             + ", ".join(f"{w:.0f}% for {c}¢" for c, w in within.items())
             + ". So the method does not invent a band where there is none, but wider bands are sometimes estimated too "
             "small. Right, the same data with the gap measured from a trailing 24-hour median: inside the band the gap "
             "wanders, the trailing median follows it, and the band disappears — which is why the main fit uses one fixed "
             "median per market.", width=160)
fig.tight_layout()
fig.savefig(OUT / "fig07_known_band.png", dpi=150, bbox_inches="tight")
plt.show()

pd.DataFrame({(KNOWN_CENTRES[cn], f"true {c}¢"): [np.median(e), np.percentile(e, 5), np.percentile(e, 95)]
              for cn in KNOWN_CENTRES for c, e in known[cn].items()},
             index=["median estimate", "5th percentile", "95th percentile"]).round(2)

## Check 3 — the c = 0 case against the textbook formulas

With no band, the model is ordinary mean reversion (an AR(1) process):
`G(t+1) = (1 − κ)·G(t) + ε`, with noise variance σ². The textbook says

* the correlation between G(t) and G(t+k) is **(1 − κ)^k**;
* the long-run variance of G is **σ² / (1 − (1 − κ)²)**;
* looking h steps ahead, the pull is **κ_h = 1 − (1 − κ)^h** (this is what the
  time-step check relies on).

First we check the code reproduces these on simulated AR(1) data, where they
must hold. Then we compare them with the real data.

In [ ]:
k_true, s_true, n_min, n_rep = 0.05, 0.003, 20000, 50
minute_ar = np.arange(n_min)
ar = simulate(minute_ar, 0.0, k_true, s_true, 0.0, 0.0, n_rep, rng, tick=None)
noflag = np.zeros(n_min, bool)
k1_hat = np.array([fit_c0(make_pairs(minute_ar, ar[:, j], noflag, 1)) for j in range(n_rep)])
k5_hat = np.array([fit_c0(make_pairs(minute_ar, ar[:, j], noflag, 5)) for j in range(n_rep)])
lags = np.arange(1, 61)
acf_sim = np.array([[np.corrcoef(ar[:-L, j], ar[L:, j])[0, 1] for L in lags] for j in range(n_rep)])
var_sim = ar.var(axis=0)
var_theory = s_true ** 2 / (1 - (1 - k_true) ** 2)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.8))
ax = axes[0]
k5_formula = 1 - (1 - k_true) ** 5
bins = np.linspace(0.85, 1.15, 25)
ax.hist(k1_hat / k_true, bins=bins, color="#9ab8e0", edgecolor="white", alpha=0.85,
        label=f"1-minute step ÷ true κ ({k_true})")
ax.hist(k5_hat / k5_formula, bins=bins, color="#f0a98a", edgecolor="white", alpha=0.85,
        label=f"5-minute step ÷ formula 1−(1−κ)⁵ ({k5_formula:.3f})")
ax.axvline(1.0, color=INK, linewidth=2, linestyle="--", label="exact agreement")
ax.set_xlabel("Fitted κ ÷ value it should have")
ax.set_ylabel("Number of simulated series")
ax.set_title("Fitted κ vs truth, 1 and 5 minutes")
ax.legend(fontsize=8, frameon=False, loc="upper left")
ax = axes[1]
ax.fill_between(lags, np.percentile(acf_sim, 5, 0), np.percentile(acf_sim, 95, 0), color="#9ab8e0", alpha=0.6,
                label="simulated (90% range)")
ax.plot(lags, (1 - k_true) ** lags, color=INK, linewidth=2, linestyle="--", label="formula (1 − κ)^k")
ax.set_xlabel("Lag k (minutes)")
ax.set_ylabel("Correlation of G(t) and G(t+k)")
ax.set_title("Autocorrelation vs formula")
ax.legend(fontsize=8, frameon=False)
ax = axes[2]
ax.hist(100 * np.sqrt(var_sim), bins=15, color="#9ab8e0", edgecolor="white", label="simulated")
ax.axvline(100 * np.sqrt(var_theory), color=INK, linewidth=2, linestyle="--", label="formula √(σ²/(1−(1−κ)²))")
ax.set_xlabel("Standard deviation of the gap (cents)")
ax.set_title("Long-run spread vs formula")
ax.legend(fontsize=8, frameon=False)
fig.suptitle("Check 3a: on simulated mean-reverting data (c = 0), the code reproduces the textbook formulas",
             fontsize=13, fontweight="bold", y=1.03)
caption(fig, f"How to read this: we simulated {n_rep} series of {n_min:,} minutes from the c = 0 model with known κ = "
             f"{k_true} and noise σ = {100 * s_true:.1f}¢, with no rounding, flicker or missing minutes. Left: the fitted κ, "
             "divided by what it should be — the true κ at a 1-minute step, 1−(1−κ)⁵ at a 5-minute step — should centre on 1. Middle: the measured "
             "autocorrelation should follow (1 − κ)^k. Right: the measured spread of the gap should match the formula. "
             "Agreement here means the fitting code is correct, so any disagreement on real data (next figure) is about "
             "the data, not the code.", width=175)
fig.tight_layout()
fig.savefig(OUT / "fig08_textbook_check_simulated.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"κ (1-min): true {k_true}, mean fitted {k1_hat.mean():.4f} ± {k1_hat.std():.4f}")
print(f"κ (5-min): formula {1 - (1 - k_true) ** 5:.4f}, mean fitted {k5_hat.mean():.4f} ± {k5_hat.std():.4f}")
print(f"spread:    formula {100 * np.sqrt(var_theory):.3f}¢, mean simulated {100 * np.sqrt(var_sim).mean():.3f}¢")
print(f"autocorrelation: largest gap between formula and simulated average over lags 1-60 = "
      f"{np.max(np.abs(acf_sim.mean(0) - (1 - k_true) ** lags)):.3f}")

### Figure 9 — the c = 0 formulas on the real data

In [ ]:
acf_lags = np.arange(1, 25)   # in 5-minute steps: 5 to 120 minutes
c0_rows = []
fig, axes = plt.subplots(3, 3, figsize=(13, 10), sharex=True, sharey=True)
for ax, label in zip(axes.flat, LABELS):
    df = EV[label]
    minute = df["minute"].to_numpy()
    dev = df["gap"].to_numpy() - centre_series(df)
    flagged = df["is_outlier"].to_numpy()
    p = PAIRS[label]
    k5 = fit_c0(p)
    acf_data = []
    for L in acf_lags:
        q = make_pairs(minute, dev, flagged, STEP * L)
        acf_data.append(np.corrcoef(q["g0"], q["g0"] + q["dg"])[0, 1])
    c0_rows.append({"event": name(label), "kappa_5min_c0": k5, "acf_30min_data": acf_data[5],
                    "acf_30min_formula": (1 - k5) ** 6, "acf_2h_data": acf_data[-1], "acf_2h_formula": (1 - k5) ** 24})
    color = CATEGORY_COLORS[CATEGORY[label]]
    ax.plot(STEP * acf_lags, (1 - k5) ** acf_lags, color=INK, linestyle="--", linewidth=1.8, label="formula (1 − κ)^k")
    ax.plot(STEP * acf_lags, acf_data, color=color, marker=CATEGORY_MARKERS[CATEGORY[label]], markersize=4, linewidth=1.5,
            label="data")
    ax.set_title(f"{name(label)} ({CATEGORY[label]})", fontsize=10)
    ax.legend(loc="upper right", fontsize=8, frameon=False)
    ax.set_ylim(-0.05, 1.02)
for ax in axes[-1]:
    ax.set_xlabel("Minutes apart")
for ax in axes[:, 0]:
    ax.set_ylabel("Correlation of the gap\nwith itself later")
fig.suptitle("Check 3b: real data vs the c = 0 formulas", fontsize=13, fontweight="bold")
caption(fig, "How to read this: the dashed line is what simple mean reversion (no band) predicts for how quickly a gap "
             "'forgets' its earlier value, using each market's fitted κ; the coloured line is what the data actually do. "
             "When the data stay correlated much longer than the formula, the gap has slow components the simple model "
             "doesn't have — for example the band (small gaps aren't pulled back at all) or a part of the gap that closes "
             "much more slowly than the rest. (The long-run variance formula is not shown for real data: with κ and the "
             "noise both estimated from the same data it matches almost by construction, so it tests nothing.)",
        width=165)
fig.tight_layout()
fig.savefig(OUT / "fig09_textbook_check_real.png", dpi=150, bbox_inches="tight")
plt.show()
c0_df = pd.DataFrame(c0_rows)
c0_df.to_csv(OUT / "c0_textbook_check.csv", index=False)
c0_df.round(3)

## Check 4 — does the model predict better than simpler models on unseen data?

Each model's κ and c are fitted on the **first 70%** of a market's time span; the
model then predicts the 5-minute change in the gap over the **last 30%**, which
it has never seen. Two questions:

**(a) Does the band help?** Same centre (the median gap of the training period), with and without the band:

| Model | Prediction of the next 5-minute change |
|---|---|
| Random walk (baseline) | 0 |
| Mean reversion, no band | −κ · (G − m) |
| **Band (main model)** | −κ · excess beyond c of (G − m) |

where m is the median gap of the first 70%.

**(b) What should the gap be measured from?** The band model with three centres:
zero, the training-period median (main), and a trailing 24-hour median (which
follows a drifting offset, but can't be used to estimate c — Figure 7).

The score is **skill**: how much smaller the squared prediction error is than
the random walk's, in %. Above 0 = better than predicting "no change". 95%
ranges come from resampling one-hour blocks of the test period.

In [ ]:
MODELS = {  # name: (centre, band?)
    "Mean reversion, no band": ("median", False),
    "Band (main model)": ("median", True),
    "Band, from zero": ("none", True),
    "Band, from trailing 24-hour median": ("rolling24h", True),
}
oos_rows, OOS_BOOT = [], {}
for label in LABELS:
    df = EV[label]
    minute, gap, flagged = df["minute"].to_numpy(), df["gap"].to_numpy(), df["is_outlier"].to_numpy()
    split = int(TRAIN_SHARE * minute[-1])
    # Every model predicts the same thing: the actual change in the gap over the next 5 minutes.
    # (The pairs are the same rows whatever the centre, since they depend only on timing and flags.)
    p_raw = make_pairs(minute, gap, flagged, STEP)
    test_rows = p_raw["minute"] >= split
    target, test_min = p_raw["dg"][test_rows], p_raw["minute"][test_rows]
    preds = {}
    for mname, (centre, band) in MODELS.items():
        dev = gap - centre_values(minute, gap, centre, train_until=split)
        p = make_pairs(minute, dev, flagged, STEP)
        train, test = p["minute"] + STEP <= split, p["minute"] >= split
        ptr = {k: v[train] for k, v in p.items()}
        g_te = p["g0"][test]
        if band:
            f = fit_threshold(ptr)
            preds[mname] = -f["kappa"] * excess(g_te, f["c"])
        else:
            preds[mname] = -fit_c0(ptr) * g_te
        assert np.array_equal(p["minute"][test], test_min)
    block = test_min // BLOCK_MINUTES
    starts = np.flatnonzero(np.r_[True, block[1:] != block[:-1]])
    W = bootstrap_weights(len(starts), N_BOOT, rng)
    row = {"label": label, "event": name(label), "n_test_pairs": len(test_min)}
    for mname in MODELS:
        sse_m = np.add.reduceat((target - preds[mname]) ** 2, starts)
        sse_rw = np.add.reduceat(target ** 2, starts)
        row[mname] = 100 * (1 - sse_m.sum() / sse_rw.sum())
        skill_b = 100 * (1 - (W @ sse_m) / (W @ sse_rw))
        row[mname + " lo"], row[mname + " hi"] = np.percentile(skill_b, [2.5, 97.5])
        OOS_BOOT[(label, mname)] = skill_b
    diff = OOS_BOOT[(label, "Band (main model)")] - OOS_BOOT[(label, "Mean reversion, no band")]
    row["band minus no band"] = row["Band (main model)"] - row["Mean reversion, no band"]
    row["band minus no band lo"], row["band minus no band hi"] = np.percentile(diff, [2.5, 97.5])
    oos_rows.append(row)
oos = pd.DataFrame(oos_rows)
oos.to_csv(OUT / "out_of_sample_skill.csv", index=False)
oos[["event", "n_test_pairs"] + list(MODELS) + ["band minus no band", "band minus no band lo", "band minus no band hi"]].round(2)

### Figure 10 — prediction skill on unseen data

In [ ]:
order = oos.merge(results[["label", "c_cents"]], on="label").sort_values("c_cents").reset_index(drop=True)
panel_a = {"Mean reversion, no band": dict(marker="o", facecolor="#9a9a9a", edgecolor="#9a9a9a"),
           "Band (main model)": dict(marker="s", facecolor=INK, edgecolor=INK)}
panel_b = {"Band, from zero": dict(marker="v", facecolor="white", edgecolor="#2a78d6"),
           "Band (main model)": dict(marker="s", facecolor=INK, edgecolor=INK),
           "Band, from trailing 24-hour median": dict(marker="D", facecolor="#9a9a9a", edgecolor="#9a9a9a")}
XMIN = -15


def draw(ax, styles):
    offs = dict(zip(styles, np.linspace(-0.2, 0.2, len(styles))))
    ax.axvline(0, color="#777777", linewidth=1)
    for i, r in order.iterrows():
        for m, st in styles.items():
            y0 = i + offs[m]
            lo, hi, v = r[m + " lo"], r[m + " hi"], r[m]
            ax.plot([max(lo, XMIN), hi], [y0, y0], color=st["edgecolor"], linewidth=1.2, alpha=0.7)
            if v < XMIN:   # off the left edge: show at the edge with its value
                ax.scatter(XMIN, y0, marker="<", color=st["edgecolor"], s=45, zorder=3)
                ax.annotate(f"{v:.0f}%", (XMIN, y0), xytext=(6, 0), textcoords="offset points", va="center",
                            fontsize=8, color=MUTED)
            else:
                ax.scatter(v, y0, marker=st["marker"], facecolor=st["facecolor"], edgecolor=st["edgecolor"], s=45, zorder=3)
    ax.set_xlim(left=XMIN)
    ax.set_yticks(range(len(order)), order["event"])
    handles = [plt.Line2D([], [], linestyle="", marker=st["marker"], markerfacecolor=st["facecolor"],
                          markeredgecolor=st["edgecolor"], markersize=8, label=m) for m, st in styles.items()]
    ax.legend(handles=handles, loc="lower center", bbox_to_anchor=(0.5, 1.0), ncol=len(styles), frameon=False, fontsize=9)


fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 7), sharey=True)
draw(ax1, panel_a)
draw(ax2, panel_b)
ax1.set_title("(a) Does the band help?", pad=34)
ax2.set_title("(b) What should the gap be measured from?", pad=34)
for ax in (ax1, ax2):
    ax.set_xlabel("Skill on the unseen last 30% (% less error than a random walk)")
fig.suptitle("Check 4: predicting the next 5-minute change on data the models never saw", fontsize=13, fontweight="bold",
             y=1.02)
caption(fig, "How to read this: every model was fitted on the first 70% of each market's history and then predicted the "
             "5-minute change in the gap over the last 30%. The horizontal axis is how much smaller its errors were than "
             "those of a random walk, which always predicts 'no change'; lines are 95% ranges. Right of the vertical line "
             "= the model genuinely predicts something. (a) Black squares (band) to the right of grey circles (no band) mean "
             "the band adds predictive power on new data, not just a closer fit to the data it was tuned on. (b) Which "
             "level to measure the gap from: zero, the median of the training period (black, main), or a trailing 24-hour "
             "median (grey). Where grey beats black, the offset between the platforms drifted during the market's life; "
             "a trailing median predicts better there but cannot be used to estimate the band (Figure 7). Values far off "
             "the left edge are marked with an arrow and their number. Skills are small because most 5-minute moves are "
             "unpredictable noise.", width=180)
fig.tight_layout()
fig.savefig(OUT / "fig10_out_of_sample.png", dpi=150, bbox_inches="tight")
plt.show()

# Part C — Appendix

## Figure A1 — robustness: other reasonable choices

In [ ]:
variants = {
    "Main: 5-min step, from median": dict(step=STEP, centre=CENTRE, keep_flagged=False),
    "1-minute step": dict(step=1, centre=CENTRE, keep_flagged=False),
    "From a trailing 24-hour median": dict(step=STEP, centre="rolling24h", keep_flagged=False),
    "From zero (no centring)": dict(step=STEP, centre="none", keep_flagged=False),
    "Flagged spikes kept in": dict(step=STEP, centre=CENTRE, keep_flagged=True),
}
vstyle = dict(zip(variants, [dict(marker="o", color=INK), dict(marker="o", color="#b9b9b9"),
                             dict(marker="D", color="#7a7a7a"), dict(marker="v", color="#2a78d6"),
                             dict(marker="x", color="#b0413e")]))
rob_rows = []
for label in LABELS:
    for vname, v in variants.items():
        f = FITS[label] if vname.startswith("Main") else fit_threshold(event_pairs(label, **v))
        rob_rows.append({"label": label, "event": name(label), "variant": vname, "c_cents": 100 * f["c"],
                         "half_life_min": float(half_life(f["kappa"], v["step"]))})
rob = pd.DataFrame(rob_rows)
rob.to_csv(OUT / "robustness.csv", index=False)

order = results.sort_values("c_cents").reset_index(drop=True)
ypos = {l: i for i, l in enumerate(order["label"])}
voff = dict(zip(variants, np.linspace(-0.3, 0.3, len(variants))))
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6.5), sharey=True)
for _, r in rob.iterrows():
    y0 = ypos[r["label"]] + voff[r["variant"]]
    st = vstyle[r["variant"]]
    ax1.scatter(r["c_cents"], y0, s=40, zorder=3, **st)
    if np.isfinite(r["half_life_min"]):
        ax2.scatter(r["half_life_min"], y0, s=40, zorder=3, **st)
ax1.set_yticks(range(len(order)), order["event"])
ax1.set_xlim(left=0)
ax1.set_xlabel("c (cents)")
ax1.set_title("Band c")
plain_log(ax2, "x")
ax2.set_xlabel("Half-life (minutes, log scale)")
ax2.set_title("Half-life")
handles = [plt.Line2D([], [], linestyle="", markersize=7, label=k, **v) for k, v in vstyle.items()]
fig.legend(handles=handles, loc="upper center", ncol=3, frameon=False, bbox_to_anchor=(0.5, 1.05))
fig.suptitle("Do the answers change with other reasonable choices?", fontsize=13, fontweight="bold", y=1.1)
caption(fig, "How to read this: black dots are the main results. Light grey: the same fit at a 1-minute step (shorter "
             "half-lives here mean flicker, see Figure 3). Dark diamonds: gaps measured from a trailing 24-hour median — "
             "this shrinks the band toward 0 even when one exists (Figure 7), so it is shown only for comparison. Blue triangles: gaps measured from zero — a steady offset between "
             "the platforms then shows up as a wide band. Red crosses: one-minute spikes kept in. Where the markers agree, "
             "the result doesn't depend on these choices.", width=170)
fig.tight_layout()
fig.savefig(OUT / "figA1_robustness.png", dpi=150, bbox_inches="tight")
plt.show()

## Figure A2 — by topic

Three markets per topic is enough to describe these markets, not to show that
topics differ in general.

In [ ]:
cat_rows = []
for cat in CATEGORY_ORDER:
    labs = [l for l in LABELS if CATEGORY[l] == cat]
    if not labs:
        continue
    c_draws = 100 * np.mean([FITS[l]["c_boot"] for l in labs], axis=0)
    hl_draws = np.exp(np.nanmean([np.log(half_life(FITS[l]["kappa_boot"])) for l in labs], axis=0))
    sub = results[results["category"] == cat]
    cat_rows.append({"category": cat, "n_markets": len(labs),
                     "mean_c_cents": sub["c_cents"].mean(), "mean_c_lo": np.percentile(c_draws, 2.5),
                     "mean_c_hi": np.percentile(c_draws, 97.5),
                     "typical_half_life_min": float(np.exp(np.log(sub["half_life_min"]).mean())),
                     "typical_hl_lo": np.nanpercentile(hl_draws, 2.5), "typical_hl_hi": np.nanpercentile(hl_draws, 97.5)})
cat_df = pd.DataFrame(cat_rows)
cat_df.to_csv(OUT / "by_topic.csv", index=False)

rows6 = []
for _, cr in cat_df.iterrows():
    rows6.append((f"{cr['category']} average", "avg", cr["category"], cr))
    for _, e in results[results["category"] == cr["category"]].sort_values("c_cents").iterrows():
        rows6.append((f"   {e['event']}", "event", cr["category"], e))
ypos6 = np.arange(len(rows6))[::-1]
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 7), sharey=True)
for yv, (text, kind, cat, r) in zip(ypos6, rows6):
    color, marker = CATEGORY_COLORS[cat], CATEGORY_MARKERS[cat]
    if kind == "avg":
        for ax in (ax1, ax2):
            ax.axhspan(yv - 0.5, yv + 0.5, color="#f3f3f1", zorder=0)
        ax1.plot([r["mean_c_lo"], r["mean_c_hi"]], [yv, yv], color=INK, linewidth=2.5, solid_capstyle="round")
        ax1.scatter(r["mean_c_cents"], yv, marker="D", color=INK, s=70, zorder=3)
        ax2.plot([r["typical_hl_lo"], r["typical_hl_hi"]], [yv, yv], color=INK, linewidth=2.5, solid_capstyle="round")
        ax2.scatter(r["typical_half_life_min"], yv, marker="D", color=INK, s=70, zorder=3)
    else:
        ax1.plot([r["c_lo_cents"], r["c_hi_cents"]], [yv, yv], color=color, linewidth=1.5, alpha=0.6)
        ax1.scatter(r["c_cents"], yv, marker=marker, color=color, s=60, edgecolor="white", linewidth=1.2, zorder=3)
        ax2.plot([r["hl_lo"], r["hl_hi"]], [yv, yv], color=color, linewidth=1.5, alpha=0.6)
        ax2.scatter(r["half_life_min"], yv, marker=marker, color=color, s=60, edgecolor="white", linewidth=1.2, zorder=3)
ax1.set_yticks(ypos6, [t for t, *_ in rows6])
for tick, (_, kind, _, _) in zip(ax1.get_yticklabels(), rows6):
    if kind == "avg":
        tick.set_fontweight("bold")
ax1.set_xlim(left=0)
ax1.set_xlabel("c: no-trade band (cents)")
ax1.set_title("How big a gap must be before it closes")
plain_log(ax2, "x")
ax2.set_xlabel("Half-life of the excess (minutes, log scale)")
ax2.set_title("How fast the excess closes")
ax1.set_ylim(-0.6, len(rows6) - 0.4)
handles = [plt.Line2D([], [], color=INK, marker="D", linestyle="-", linewidth=2.5, markersize=7, label="topic average (95% range)")]
handles += [plt.Line2D([], [], color=CATEGORY_COLORS[c], marker=CATEGORY_MARKERS[c], linestyle="-", alpha=0.8, markersize=7,
                       label=f"{c} market (95% range)") for c in CATEGORY_ORDER]
fig.legend(handles=handles, loc="upper center", ncol=4, frameon=False, bbox_to_anchor=(0.5, 1.02))
fig.suptitle("By topic: the three markets in each topic and their average", fontsize=13, fontweight="bold", y=1.07)
caption(fig, "How to read this: shaded rows are topic averages (black diamond, 95% range as a thick bar; for half-lives the "
             "average is taken on a log scale, i.e. a typical value); the rows under each are that topic's markets. Left: "
             "the band c. Right: how many minutes the excess beyond c takes to halve. With three markets per topic this "
             "describes these nine markets; it is not enough to prove that topics differ in general.", width=170)
fig.tight_layout()
fig.savefig(OUT / "figA2_by_topic.png", dpi=150, bbox_inches="tight")
plt.show()
cat_df.round(2)

## Figure A3 — liquidity and activity

In [ ]:
def spearman(x, y):
    return pd.Series(np.asarray(x)).rank().corr(pd.Series(np.asarray(y)).rank())


measures = [("kalshi_contracts_per_sec", "Kalshi contracts\ntraded per second", True),
            ("kalshi_trade_minute_share", "Share of minutes\nwith a Kalshi trade", False),
            ("kalshi_spread_cents", "Kalshi bid–ask\nspread (cents)", False),
            ("poly_move_share", "Share of minutes\nPolymarket price moves", False)]
targets = [("c_cents", "c (cents)", False), ("half_life_min", "Half-life (minutes,\nlog scale)", True)]
fig, axes = plt.subplots(2, 4, figsize=(16, 8.5))
corr_rows, panel_texts = [], []
for row, (ycol, ylabel, ylog) in enumerate(targets):
    for col, (xcol, xlabel, xlog) in enumerate(measures):
        ax = axes[row, col]
        texts = []
        for _, r in results.iterrows():
            cat = r["category"]
            ax.scatter(r[xcol], r[ycol], color=CATEGORY_COLORS[cat], marker=CATEGORY_MARKERS[cat], s=60,
                       edgecolor="white", linewidth=1.2, zorder=3)
            texts.append(ax.annotate(r["event"], (r[xcol], r[ycol]), xytext=(5, 0), textcoords="offset points",
                                     va="center", fontsize=7, color="#555555", arrowprops=LEADER))
        if xlog:
            plain_log(ax, "x", subs=(1.0,))
        if ylog:
            plain_log(ax, "y")
        rho = spearman(results[xcol], results[ycol])
        corr_rows.append({"y": ycol, "x": xcol, "spearman_rho": rho})
        ax.set_title(f"rank correlation ρ = {rho:+.2f}", fontsize=9.5, fontweight="normal")
        if row == 1:
            ax.set_xlabel(xlabel)
        if col == 0:
            ax.set_ylabel(ylabel)
        panel_texts.append((ax, texts))
fig.legend(handles=category_handles(), loc="upper center", ncol=3, frameon=False, bbox_to_anchor=(0.5, 1.02))
fig.suptitle("Are the band and the half-life related to how liquid and active each market is?", fontsize=13,
             fontweight="bold", y=1.06)
fig.tight_layout()
for ax, texts in panel_texts:
    dodge_labels(ax, texts)
caption(fig, "How to read this: top row — the band c; bottom row — the half-life. Each column is one measure of how liquid "
             "or active a market is. ρ is the rank correlation across the nine markets: +1 means the two always rise "
             "together, −1 means one falls as the other rises, 0 means no relation. With nine markets only |ρ| above about "
             "0.7 is worth taking seriously.", width=190)
fig.savefig(OUT / "figA3_liquidity.png", dpi=150, bbox_inches="tight")
plt.show()
corr = pd.DataFrame(corr_rows)
corr.to_csv(OUT / "liquidity_correlations.csv", index=False)
results.to_csv(OUT / "main_fit_by_market.csv", index=False)

# Summary

Generated from the numbers above, so it stays correct if the data change.

In [ ]:
L = []
L.append("MAIN FIT (5-minute step, gaps measured from each market's median gap)")
for _, r in results.sort_values(["category", "c_cents"]).iterrows():
    L.append(f"  {r['category']:9s} {r['event']:20s} c = {r['c_cents']:4.1f}¢ ({r['c_lo_cents']:.1f}–{r['c_hi_cents']:.1f})"
             f"   half-life {fmt_minutes(r['half_life_min'])} ({fmt_minutes(r['hl_lo'])}–{fmt_minutes(r['hl_hi'])})"
             f"   Kalshi-side cost {r['kalshi_cost_cents']:.1f}¢")

L += ["", "CHECK 1 — time step (half-life in minutes at 1 / 5 / 15-minute steps)"]
piv = steps_df.pivot(index="label", columns="step", values="half_life_min")
STEP_VERDICT = {}
for label in LABELS:
    h1, h5, h15 = piv.loc[label, 1], piv.loc[label, 5], piv.loc[label, 15]
    r1, r2 = h5 / h1, h15 / h5
    if not (np.isfinite(r1) and np.isfinite(r2)):
        verdict = "cannot compare"
    elif 0.5 <= r1 <= 2 and 0.5 <= r2 <= 2:
        verdict = "consistent"
    elif r1 > 2 and r2 > 2:
        verdict = "keeps slowing at longer steps (fast part + slow leftover)"
    elif r1 > 2:
        verdict = "1-minute much faster (flicker); 5 and 15 agree"
    elif r2 > 2:
        verdict = "15-minute much slower (slow leftover)"
    else:
        verdict = "inconsistent"
    STEP_VERDICT[label] = verdict
    L.append(f"  {name(label):20s} {fmt_minutes(h1):>8s} / {fmt_minutes(h5):>8s} / {fmt_minutes(h15):>8s}   {verdict}")

L += ["", "CHECK 2 — simulation (does the fit recover known values?)"]
for label in LABELS:
    s = SIM[label]
    c_ok, hl_ok = s["c_cov"] >= 0.85, s["hl_cov"] >= 0.85
    verdict = ("both recovered" if c_ok and hl_ok else
               "c recovered; half-life off" if c_ok else
               "half-life recovered; c off" if hl_ok else "neither recovered reliably")
    L.append(f"  {name(label):20s} c: true {100 * s['c_true']:.1f}¢, median estimate {100 * np.median(s['c_hat']):.1f}¢, "
             f"95% range covers truth {100 * s['c_cov']:.0f}%;  half-life: true {fmt_minutes(s['hl_true'])}, "
             f"median estimate {fmt_minutes(np.nanmedian(s['hl_hat']))}, covers {100 * s['hl_cov']:.0f}%   -> {verdict}")
for cn, title in KNOWN_CENTRES.items():
    L.append(f"  Known bands, {title.lower()} (template {name(template)}): " +
             ", ".join(f"true {c}¢ → {np.median(e):.1f}¢" for c, e in known[cn].items()))

L += ["", "CHECK 3 — c = 0 textbook formulas",
      f"  simulated: κ true {k_true} vs fitted {k1_hat.mean():.4f}; 5-min formula {1 - (1 - k_true) ** 5:.4f} vs fitted "
      f"{k5_hat.mean():.4f}; spread formula {100 * np.sqrt(var_theory):.3f}¢ vs {100 * np.sqrt(var_sim).mean():.3f}¢"]
for _, r in c0_df.iterrows():
    L.append(f"  {r['event']:20s} autocorrelation after 30 min: data {r['acf_30min_data']:.2f} vs formula "
             f"{r['acf_30min_formula']:.2f};  after 2 hours: data {r['acf_2h_data']:.2f} vs formula {r['acf_2h_formula']:.2f}")

L += ["", "CHECK 4 — out-of-sample skill vs random walk (%, last 30% of each market)"]
for _, r in oos.iterrows():
    d_lo, d_hi = r["band minus no band lo"], r["band minus no band hi"]
    sig = "band better" if d_lo > 0 else ("band worse" if d_hi < 0 else "no clear difference")
    L.append(f"  {r['event']:20s} band (main) {r['Band (main model)']:+5.1f} ({r['Band (main model) lo']:+.1f} to "
             f"{r['Band (main model) hi']:+.1f});  no band {r['Mean reversion, no band']:+5.1f};  "
             f"band vs no band: {sig};  from zero {r['Band, from zero']:+.1f}, from trailing 24h median "
             f"{r['Band, from trailing 24-hour median']:+.1f}")
print("\n".join(L))
open(OUT / "summary.txt", "w", encoding="utf-8").write("\n".join(L))

# Which results can be trusted? One row per market, combining the four checks.
trust = []
for label in LABELS:
    r, s_, o = R.loc[label], SIM[label], oos.set_index("label").loc[label]
    c_bias = 100 * (np.median(s_["c_hat"]) - s_["c_true"])
    hl_ratio = np.nanmedian(s_["hl_hat"]) / s_["hl_true"]
    predicts = o["Band (main model) lo"] > 0
    band_helps = o["band minus no band lo"] > 0
    unbiased = abs(c_bias) <= 0.3 and 0.5 <= hl_ratio <= 2
    if r["c_lo_cents"] <= 0.0:
        overall = "no clear band (its 95% range reaches 0¢)"
    elif predicts and band_helps and unbiased:
        overall = "supported"
    elif predicts and band_helps:
        overall = "band helps prediction; size/speed biased (check 2)"
    elif predicts:
        overall = "predictable, but the band isn't shown to help"
    elif o["Band, from trailing 24-hour median lo"] > 0:
        overall = "offset drifted: band predicts only with a moving centre (check 4b)"
    else:
        overall = "not supported"
    trust.append({"market": name(label), "c (¢)": f"{r['c_cents']:.1f} ({r['c_lo_cents']:.1f}–{r['c_hi_cents']:.1f})",
                  "half-life": fmt_minutes(r["half_life_min"]),
                  "1: time step": STEP_VERDICT[label],
                  "2: sim bias in c (¢)": round(c_bias, 1),
                  "2: sim half-life / truth": round(hl_ratio, 1),
                  "4: beats random walk": "yes" if predicts else "no",
                  "4: band beats no band": "yes" if band_helps else "no",
                  "overall": overall})
trust_df = pd.DataFrame(trust)
trust_df.to_csv(OUT / "trust_table.csv", index=False)
print("\nWHICH RESULTS CAN BE TRUSTED  (check 3 tests the code and the c = 0 model, so it isn't per market)")
print("  sim bias in c: median estimate minus truth on simulated data (negative = the method underestimates c)")
print("  sim half-life / truth: above 1 = the method makes closing look slower than it is")
with pd.option_context("display.max_colwidth", 60, "display.width", 250):
    display(trust_df)
print(f"\nTables and figures saved in {OUT.resolve()}  (total run time {time.time() - T_START:.0f}s)")

### Caveats

* **Nine markets.** Enough to describe these markets, not to generalise about topics.
* **Kalshi-side cost** uses Kalshi's standard fee rate (`KALSHI_FEE_RATE`) and
  ignores the Polymarket side (no historical Polymarket spreads are available), so
  it is a lower bound on the real cost of closing a gap.
* **Simulation realism.** The fake data copy the timeline, rounding, flicker and
  stale quotes, but not everything (e.g. news jumps or a drifting offset). Passing
  the simulation check is necessary, not sufficient.
* **The median centre** assumes the offset between platforms is constant over a
  market's life; Appendix A1 shows a rolling 24-hour version.
* **Out-of-sample split.** The last 30% of each market is close to resolution, when
  prices move more; skills there may differ from earlier periods.